# `mart_events` 

Полный `train_clean` содержит около 100 млн событий. Запрос с несколькими конструкциями

`PARTITION BY user_id ORDER BY timestamp, row_id`

заставляет DuckDB сортировать очень большой объём данных. При лимите RAM 4 GB большая часть сортировки уходит на диск.

Дополнительно исходная версия:

- отдельно сортирует весь event log;
- отдельно сортирует все question events;
- ещё раз сортирует итог перед записью;
- затем повторяет тяжёлые окна в полном leakage-check.

## Новый алгоритм

1. Один раз читаем `train_clean.parquet`.
2. Делим данные на `N_PARTITIONS = 32` частей по `hash(user_id)`.
3. Все события одного пользователя всегда попадают в одну часть.
4. Последовательностные признаки считаются независимо внутри каждой маленькой части.
5. Каждая часть сортируется по `user_id, timestamp, row_id`.
6. Готовые части последовательно объединяются в один `mart_events.parquet`.
7. Leakage проверяется на одной полной части пользователей, а не повторным проходом по 100 млн строк.

Таким образом вместо одной огромной сортировки получаем несколько существенно меньших сортировок, которые гораздо реже уходят в disk spill.

> Цель — заметно сократить время расчёта. Гарантировать строго `<20 минут` на любом компьютере невозможно: итог зависит от SSD, CPU и RAM. На SSD и обычном современном ноутбуке такой алгоритм должен быть существенно быстрее исходного.

## 1. Настройки

In [2]:
# Если библиотек нет:
# !pip install duckdb pandas pyarrow psutil

from pathlib import Path
import os
import shutil
import time

import duckdb
import pandas as pd
import pyarrow.parquet as pq

# Ноутбук может лежать либо в корне проекта, либо внутри data/.
if (Path("processed") / "train_clean.parquet").exists():
    DATA_DIR = Path(".")
elif (Path("data") / "processed" / "train_clean.parquet").exists():
    DATA_DIR = Path("data")
else:
    raise FileNotFoundError(
        "Не найден train_clean.parquet. "
        "Запусти ноутбук из корня проекта или из папки data."
    )

PROCESSED_DIR = DATA_DIR / "processed"
CACHE_DIR = DATA_DIR / "mart_cache"
TEMP_DIR = DATA_DIR / "tmp_duckdb"

TRAIN_PATH = PROCESSED_DIR / "train_clean.parquet"
QUESTIONS_PATH = PROCESSED_DIR / "questions_clean.parquet"
LECTURES_PATH = PROCESSED_DIR / "lectures_clean.parquet"
MART_PATH = PROCESSED_DIR / "mart_events.parquet"

# Чем больше частей, тем меньше каждая оконная сортировка.
# 32 — хороший старт для ноутбука с 8–32 GB RAM.
N_PARTITIONS = 32

TRAIN_PARTS_DIR = CACHE_DIR / f"train_parts_{N_PARTITIONS}"
MART_PARTS_DIR = CACHE_DIR / f"mart_parts_{N_PARTITIONS}"

SESSION_GAP_MINUTES = 30
SESSION_GAP_MS = SESSION_GAP_MINUTES * 60 * 1000

# Если предыдущий запуск оборвался, готовые mart_part_*.parquet
# будут автоматически пропущены.
RESUME = True

# Быстрый Parquet-кодек. Если важнее минимальный размер,
# после окончания можно заменить на zstd.
FINAL_COMPRESSION = "snappy"

for path in [TRAIN_PATH, QUESTIONS_PATH, LECTURES_PATH]:
    assert path.exists(), f"Не найден файл: {path.resolve()}"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)
MART_PARTS_DIR.mkdir(parents=True, exist_ok=True)

def sql_path(path: Path) -> str:
    return str(path.resolve()).replace("\\", "/").replace("'", "''")

TRAIN = sql_path(TRAIN_PATH)
QUESTIONS = sql_path(QUESTIONS_PATH)
LECTURES = sql_path(LECTURES_PATH)
TEMP = sql_path(TEMP_DIR)

# Автоматически используем больше ресурсов, чем в старой версии,
# но не забираем весь компьютер.
cpu_count = os.cpu_count() or 4
DUCKDB_THREADS = min(8, max(4, cpu_count))

try:
    import psutil
    total_ram_gb = psutil.virtual_memory().total / 1024**3
    memory_gb = max(2, min(8, int(total_ram_gb * 0.55)))
except Exception:
    total_ram_gb = None
    memory_gb = 4

DUCKDB_MEMORY_LIMIT = f"{memory_gb}GB"

con = duckdb.connect()
con.execute(f"SET threads = {DUCKDB_THREADS}")
con.execute(f"SET memory_limit = '{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"SET temp_directory = '{TEMP}'")
con.execute("SET preserve_insertion_order = false")

print("DuckDB:", duckdb.__version__)
print("Threads:", DUCKDB_THREADS)
print("Memory limit:", DUCKDB_MEMORY_LIMIT)
if total_ram_gb is not None:
    print(f"RAM компьютера: {total_ram_gb:.1f} GB")
print("User partitions:", N_PARTITIONS)
print("Session gap:", SESSION_GAP_MINUTES, "min")

DuckDB: 1.5.5
Threads: 8
Memory limit: 8GB
RAM компьютера: 31.6 GB
User partitions: 32
Session gap: 30 min


## 2. Маленькие справочники держим в памяти

`questions_clean` и `lectures_clean` очень маленькие относительно event log, поэтому материализуем их один раз.

Заодно проверяем уникальность ключей — это защищает JOIN от размножения строк.

In [3]:
con.execute("DROP TABLE IF EXISTS dim_questions")
con.execute("DROP TABLE IF EXISTS dim_lectures")

con.execute(f"""
CREATE TEMP TABLE dim_questions AS
SELECT *
FROM read_parquet('{QUESTIONS}')
""")

con.execute(f"""
CREATE TEMP TABLE dim_lectures AS
SELECT *
FROM read_parquet('{LECTURES}')
""")

dq_dims = con.sql("""
SELECT
    (SELECT COUNT(*) FROM dim_questions) AS question_rows,
    (SELECT COUNT(DISTINCT question_id) FROM dim_questions) AS question_ids,
    (SELECT COUNT(*) FROM dim_lectures) AS lecture_rows,
    (SELECT COUNT(DISTINCT lecture_id) FROM dim_lectures) AS lecture_ids
""").df()

display(dq_dims)

r = dq_dims.iloc[0]
assert int(r.question_rows) == int(r.question_ids)
assert int(r.lecture_rows) == int(r.lecture_ids)

SOURCE_ROWS = con.sql(
    f"SELECT COUNT(*) FROM read_parquet('{TRAIN}')"
).fetchone()[0]

print(f"✅ Source events: {SOURCE_ROWS:,}")

,question_rows,question_ids,lecture_rows,lecture_ids
0,13523,13523,418,418


✅ Source events: 101,230,332


## 3. Один раз разбиваем `train_clean` по пользователям

Ключевой шаг оптимизации.

Используем:

`bucket = hash(user_id) % 32`

Поэтому **все события одного `user_id` находятся в одном bucket**. Последовательностные признаки можно считать независимо по каждой части.

Эта стадия читает исходный `train_clean.parquet` только один раз.

Если папка уже существует после предыдущего запуска, она переиспользуется.

In [4]:
partition_started = time.perf_counter()

expected_bucket_dirs = [
    TRAIN_PARTS_DIR / f"bucket={i}"
    for i in range(N_PARTITIONS)
]

partitions_ready = (
    TRAIN_PARTS_DIR.exists()
    and all(
        d.exists() and any(d.glob("*.parquet"))
        for d in expected_bucket_dirs
    )
)

if partitions_ready and RESUME:
    print("✅ Train partitions уже существуют — повторное разбиение пропускаем.")
else:
    if TRAIN_PARTS_DIR.exists():
        shutil.rmtree(TRAIN_PARTS_DIR)

    TRAIN_PARTS_DIR.mkdir(parents=True, exist_ok=True)
    train_parts_sql = sql_path(TRAIN_PARTS_DIR)

    print("Разбиваем train_clean по hash(user_id)...")

    con.execute(f"""
    COPY (
        SELECT
            *,
            CAST(hash(user_id) % {N_PARTITIONS} AS INTEGER) AS bucket
        FROM read_parquet('{TRAIN}')
    )
    TO '{train_parts_sql}'
    (
        FORMAT PARQUET,
        PARTITION_BY (bucket),
        COMPRESSION ZSTD,
        COMPRESSION_LEVEL 1,
        OVERWRITE_OR_IGNORE
    )
    """)

elapsed = time.perf_counter() - partition_started
print(f"✅ Partition stage: {elapsed / 60:.2f} min")

Разбиваем train_clean по hash(user_id)...
✅ Partition stage: 0.10 min


## 4. Функция построения одной части

Внутри bucket обычно находится лишь несколько процентов всех событий.

### Оптимизации внутри запроса

- bucket читается один раз через `AS MATERIALIZED`;
- streak не требует трёх дополнительных сортировок;
- серия считается через номер последнего противоположного ответа;
- JOIN-ы идут с маленькими in-memory dimensions;
- нет глобального `ORDER BY` на 100 млн строк.

### Leakage

Question snapshot с номером `k` описывает состояние **после k-го вопроса**.

Текущее событие соединяется со snapshot:

`questions_before = k`

поэтому ответ текущего вопроса не может попасть в его исторические признаки.

In [5]:
def build_partition(bucket: int):
    bucket_dir = TRAIN_PARTS_DIR / f"bucket={bucket}"
    input_glob = sql_path(bucket_dir / "*.parquet")

    output_path = MART_PARTS_DIR / f"mart_part_{bucket:02d}.parquet"
    output_sql = sql_path(output_path)

    if RESUME and output_path.exists() and output_path.stat().st_size > 0:
        return output_path, None, "skipped"

    if output_path.exists():
        output_path.unlink()

    sql = f"""
    COPY (
    WITH
    src AS MATERIALIZED (
        SELECT *
        FROM read_parquet(
            '{input_glob}',
            hive_partitioning = false
        )
    ),

    event_w1 AS (
        SELECT
            *,

            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY timestamp, row_id
            ) AS event_number,

            LAG(timestamp) OVER (
                PARTITION BY user_id
                ORDER BY timestamp, row_id
            ) AS previous_timestamp,

            SUM(CASE WHEN content_type_id = 0 THEN 1 ELSE 0 END) OVER (
                PARTITION BY user_id
                ORDER BY timestamp, row_id
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )
            - CASE WHEN content_type_id = 0 THEN 1 ELSE 0 END
            AS questions_before,

            SUM(CASE WHEN content_type_id = 1 THEN 1 ELSE 0 END) OVER (
                PARTITION BY user_id
                ORDER BY timestamp, row_id
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )
            - CASE WHEN content_type_id = 1 THEN 1 ELSE 0 END
            AS lectures_before,

            SUM(
                CASE
                    WHEN content_type_id = 0
                     AND answered_correctly = 1
                    THEN 1 ELSE 0
                END
            ) OVER (
                PARTITION BY user_id
                ORDER BY timestamp, row_id
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )
            - CASE
                WHEN content_type_id = 0
                 AND answered_correctly = 1
                THEN 1 ELSE 0
              END
            AS correct_answers_before

        FROM src
    ),

    event_w2 AS (
        SELECT
            *,

            timestamp - previous_timestamp AS time_since_last_event,

            CASE
                WHEN content_type_id = 0
                THEN questions_before + 1
                ELSE NULL
            END AS question_number,

            CASE
                WHEN questions_before = 0 THEN NULL
                ELSE correct_answers_before * 1.0 / questions_before
            END AS user_accuracy_before,

            CASE
                WHEN previous_timestamp IS NULL THEN 1
                WHEN timestamp - previous_timestamp > {SESSION_GAP_MS} THEN 1
                ELSE 0
            END AS new_session_flag

        FROM event_w1
    ),

    events AS (
        SELECT
            *,

            SUM(new_session_flag) OVER (
                PARTITION BY user_id
                ORDER BY timestamp, row_id
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS session_id

        FROM event_w2
    ),

    q0 AS (
        SELECT
            user_id,
            timestamp,
            row_id,
            answered_correctly,

            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY timestamp, row_id
            ) AS question_number

        FROM src
        WHERE content_type_id = 0
    ),

    q1 AS (
        SELECT
            *,

            AVG(answered_correctly) OVER (
                PARTITION BY user_id
                ORDER BY timestamp, row_id
                ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
            ) AS rolling_accuracy_5_after,

            AVG(answered_correctly) OVER (
                PARTITION BY user_id
                ORDER BY timestamp, row_id
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS rolling_accuracy_20_after,

            MAX(
                CASE
                    WHEN answered_correctly = 1
                    THEN question_number
                    ELSE 0
                END
            ) OVER (
                PARTITION BY user_id
                ORDER BY timestamp, row_id
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS last_correct_question,

            MAX(
                CASE
                    WHEN answered_correctly = 0
                    THEN question_number
                    ELSE 0
                END
            ) OVER (
                PARTITION BY user_id
                ORDER BY timestamp, row_id
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS last_wrong_question

        FROM q0
    ),

    snapshots AS (
        SELECT
            user_id,
            question_number,

            answered_correctly AS previous_correct,
            rolling_accuracy_5_after AS rolling_accuracy_5,
            rolling_accuracy_20_after AS rolling_accuracy_20,

            CASE
                WHEN answered_correctly = 0
                THEN question_number - last_correct_question
                ELSE 0
            END AS error_streak,

            CASE
                WHEN answered_correctly = 1
                THEN question_number - last_wrong_question
                ELSE 0
            END AS correct_streak

        FROM q1
    ),

    mart AS (
        SELECT
            e.row_id,
            e.user_id,
            e.timestamp,
            e.content_id,
            e.content_type_id,

            CASE
                WHEN e.content_type_id = 0 THEN 'question'
                ELSE 'lecture'
            END AS content_kind,

            e.task_container_id,

            CASE
                WHEN e.content_type_id = 0 THEN e.user_answer
                ELSE NULL
            END AS user_answer,

            CASE
                WHEN e.content_type_id = 0 THEN e.answered_correctly
                ELSE NULL
            END AS answered_correctly,

            e.prior_question_elapsed_time,
            e.prior_question_had_explanation,

            q.question_id,
            q.bundle_id,
            q.correct_answer,
            q.part AS question_part,
            q.tags,

            l.lecture_id,
            l.tag,
            l.part AS lecture_part,
            l.type_of,

            COALESCE(q.part, l.part) AS part,

            e.event_number,
            e.question_number,

            s.previous_correct,
            s.rolling_accuracy_5,
            s.rolling_accuracy_20,

            e.time_since_last_event,
            e.lectures_before,
            e.questions_before,
            e.correct_answers_before,
            e.user_accuracy_before,

            COALESCE(s.error_streak, 0) AS error_streak,
            COALESCE(s.correct_streak, 0) AS correct_streak,

            e.session_id

        FROM events e

        LEFT JOIN dim_questions q
            ON e.content_type_id = 0
           AND e.content_id = q.question_id

        LEFT JOIN dim_lectures l
            ON e.content_type_id = 1
           AND e.content_id = l.lecture_id

        LEFT JOIN snapshots s
            ON e.user_id = s.user_id
           AND e.questions_before = s.question_number
    )

    SELECT *
    FROM mart
    ORDER BY user_id, timestamp, row_id
    )
    TO '{output_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD,
        COMPRESSION_LEVEL 1,
        ROW_GROUP_SIZE 250000
    )
    """

    started = time.perf_counter()
    con.execute(sql)
    elapsed = time.perf_counter() - started

    rows = con.sql(
        f"SELECT COUNT(*) FROM read_parquet('{output_sql}')"
    ).fetchone()[0]

    return output_path, rows, elapsed

## 5. Считаем 32 независимые части

In [6]:
build_started = time.perf_counter()

part_files = []
new_rows = 0
skipped = 0

for bucket in range(N_PARTITIONS):
    part, rows, elapsed = build_partition(bucket)
    part_files.append(part)

    if elapsed == "skipped":
        skipped += 1
        print(
            f"[{bucket + 1:02d}/{N_PARTITIONS}] "
            f"готово ранее → {part.name}"
        )
    else:
        new_rows += rows
        print(
            f"[{bucket + 1:02d}/{N_PARTITIONS}] "
            f"{rows:,} rows | {elapsed:.1f} sec"
        )

total_elapsed = time.perf_counter() - build_started

print()
print(f"✅ Parts stage: {total_elapsed / 60:.2f} min")
print(f"Пропущено готовых частей: {skipped}")

[01/32] 3,120,880 rows | 24.2 sec
[02/32] 3,280,228 rows | 26.3 sec
[03/32] 3,241,620 rows | 24.3 sec
[04/32] 3,128,117 rows | 23.6 sec
[05/32] 3,158,933 rows | 23.6 sec
[06/32] 3,141,041 rows | 24.9 sec
[07/32] 3,111,336 rows | 24.8 sec
[08/32] 3,183,251 rows | 25.5 sec
[09/32] 3,235,726 rows | 26.1 sec
[10/32] 3,260,090 rows | 26.6 sec
[11/32] 3,323,974 rows | 26.2 sec
[12/32] 3,180,217 rows | 26.6 sec
[13/32] 3,287,535 rows | 26.7 sec
[14/32] 3,195,344 rows | 26.2 sec
[15/32] 3,166,097 rows | 26.5 sec
[16/32] 2,993,313 rows | 24.4 sec
[17/32] 3,037,998 rows | 25.3 sec
[18/32] 3,064,998 rows | 24.3 sec
[19/32] 3,301,385 rows | 26.2 sec
[20/32] 3,120,051 rows | 24.6 sec
[21/32] 3,131,110 rows | 27.1 sec
[22/32] 3,384,476 rows | 29.7 sec
[23/32] 3,149,001 rows | 27.3 sec
[24/32] 3,091,159 rows | 25.4 sec
[25/32] 3,026,477 rows | 24.7 sec
[26/32] 3,141,520 rows | 27.2 sec
[27/32] 3,060,901 rows | 26.6 sec
[28/32] 3,115,248 rows | 25.5 sec
[29/32] 3,143,869 rows | 26.3 sec
[30/32] 3,132,

## 6. Быстрые DQ-проверки по частям

Проверки выполняются по маленьким parquet-файлам и не требуют новой глобальной сортировки.

Проверяем:

- число строк;
- уникальность `row_id`;
- разделение question/lecture events;
- корректность JOIN metadata;
- диапазоны accuracy.

In [7]:
dq_started = time.perf_counter()

total_rows = 0
total_duplicates = 0
lecture_with_answer = 0
question_without_metadata = 0
lecture_without_metadata = 0
bad_accuracy = 0

for part in part_files:
    p = sql_path(part)

    r = con.sql(f"""
    SELECT
        COUNT(*) AS n,
        COUNT(*) - COUNT(DISTINCT row_id) AS duplicate_rows,

        COUNT(*) FILTER (
            WHERE content_type_id = 1
              AND answered_correctly IS NOT NULL
        ) AS lecture_with_answer,

        COUNT(*) FILTER (
            WHERE content_type_id = 0
              AND question_id IS NULL
        ) AS question_without_metadata,

        COUNT(*) FILTER (
            WHERE content_type_id = 1
              AND lecture_id IS NULL
        ) AS lecture_without_metadata,

        COUNT(*) FILTER (
            WHERE
                (rolling_accuracy_5 IS NOT NULL
                 AND NOT rolling_accuracy_5 BETWEEN 0 AND 1)
                OR
                (rolling_accuracy_20 IS NOT NULL
                 AND NOT rolling_accuracy_20 BETWEEN 0 AND 1)
                OR
                (user_accuracy_before IS NOT NULL
                 AND NOT user_accuracy_before BETWEEN 0 AND 1)
        ) AS bad_accuracy

    FROM read_parquet('{p}')
    """).df().iloc[0]

    total_rows += int(r.n)
    total_duplicates += int(r.duplicate_rows)
    lecture_with_answer += int(r.lecture_with_answer)
    question_without_metadata += int(r.question_without_metadata)
    lecture_without_metadata += int(r.lecture_without_metadata)
    bad_accuracy += int(r.bad_accuracy)

assert total_rows == SOURCE_ROWS, (
    f"Количество строк изменилось: {SOURCE_ROWS:,} -> {total_rows:,}"
)
assert total_duplicates == 0
assert lecture_with_answer == 0
assert question_without_metadata == 0
assert lecture_without_metadata == 0
assert bad_accuracy == 0

print(f"Rows: {total_rows:,}")
print("Duplicates:", total_duplicates)
print("Lecture rows with answer:", lecture_with_answer)
print("Questions without metadata:", question_without_metadata)
print("Lectures without metadata:", lecture_without_metadata)
print("Bad accuracy values:", bad_accuracy)
print(f"✅ Fast DQ: {(time.perf_counter() - dq_started) / 60:.2f} min")

Rows: 101,230,332
Duplicates: 0
Lecture rows with answer: 0
Questions without metadata: 0
Lectures without metadata: 0
Bad accuracy values: 0
✅ Fast DQ: 0.07 min


## 7. Leakage-check без повторного прохода по 100 млн строк

Проверяем **одну полную user-partition**, а не случайные отдельные строки.

Это важно: для выбранных пользователей присутствует вся их история, поэтому rolling-признаки можно пересчитать корректно.

Текущий ответ исключается окном, заканчивающимся на `1 PRECEDING`.

In [8]:
leak_started = time.perf_counter()

# Берём первую полную partition.
sample_part = sql_path(part_files[0])

leakage_dq = con.sql(f"""
WITH q AS (
    SELECT
        user_id,
        question_number,
        answered_correctly,
        previous_correct,
        rolling_accuracy_5,
        rolling_accuracy_20,

        LAG(answered_correctly) OVER (
            PARTITION BY user_id
            ORDER BY question_number
        ) AS expected_previous_correct,

        AVG(answered_correctly) OVER (
            PARTITION BY user_id
            ORDER BY question_number
            ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
        ) AS expected_rolling_5,

        AVG(answered_correctly) OVER (
            PARTITION BY user_id
            ORDER BY question_number
            ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING
        ) AS expected_rolling_20

    FROM read_parquet('{sample_part}')
    WHERE content_type_id = 0
)

SELECT
    COUNT(*) AS checked_questions,

    COUNT(*) FILTER (
        WHERE previous_correct
              IS DISTINCT FROM expected_previous_correct
    ) AS bad_previous_correct,

    COUNT(*) FILTER (
        WHERE
            ((rolling_accuracy_5 IS NULL)
             <> (expected_rolling_5 IS NULL))
            OR (
                rolling_accuracy_5 IS NOT NULL
                AND expected_rolling_5 IS NOT NULL
                AND ABS(rolling_accuracy_5 - expected_rolling_5) > 1e-12
            )
    ) AS bad_rolling_5,

    COUNT(*) FILTER (
        WHERE
            ((rolling_accuracy_20 IS NULL)
             <> (expected_rolling_20 IS NULL))
            OR (
                rolling_accuracy_20 IS NOT NULL
                AND expected_rolling_20 IS NOT NULL
                AND ABS(rolling_accuracy_20 - expected_rolling_20) > 1e-12
            )
    ) AS bad_rolling_20

FROM q
""").df()

display(leakage_dq)

r = leakage_dq.iloc[0]

assert int(r.bad_previous_correct) == 0
assert int(r.bad_rolling_5) == 0
assert int(r.bad_rolling_20) == 0

print(
    f"✅ Leakage-check: {int(r.checked_questions):,} questions, "
    f"{time.perf_counter() - leak_started:.1f} sec"
)

,checked_questions,bad_previous_correct,bad_rolling_5,bad_rolling_20
0,3061561,0,0,0


✅ Leakage-check: 3,061,561 questions, 0.4 sec


## 8. Объединяем части в один `mart_events.parquet`

Важно: здесь **нет глобальной сортировки**.

Каждый пользователь целиком принадлежит одному bucket, а внутри bucket данные уже записаны в порядке:

`user_id, timestamp, row_id`.

Мы просто последовательно переносим row groups из `mart_part_00`, затем `mart_part_01` и т.д.

Поэтому порядок событий **внутри каждого пользователя сохраняется**.

In [9]:
merge_started = time.perf_counter()

if MART_PATH.exists():
    MART_PATH.unlink()

writer = None

try:
    for i, part in enumerate(part_files, start=1):
        pf = pq.ParquetFile(part)

        if writer is None:
            writer = pq.ParquetWriter(
                MART_PATH,
                pf.schema_arrow,
                compression=FINAL_COMPRESSION,
                use_dictionary=True
            )

        for rg in range(pf.num_row_groups):
            table = pf.read_row_group(rg)

            if not table.schema.equals(
                writer.schema,
                check_metadata=False
            ):
                table = table.cast(writer.schema)

            writer.write_table(table)

        print(f"[{i:02d}/{N_PARTITIONS}] merged {part.name}")

finally:
    if writer is not None:
        writer.close()

merge_elapsed = time.perf_counter() - merge_started

print(f"✅ Merge stage: {merge_elapsed / 60:.2f} min")
print("Result:", MART_PATH.resolve())
print(f"Size: {MART_PATH.stat().st_size / 1024**3:.2f} GB")

[01/32] merged mart_part_00.parquet
[02/32] merged mart_part_01.parquet
[03/32] merged mart_part_02.parquet
[04/32] merged mart_part_03.parquet
[05/32] merged mart_part_04.parquet
[06/32] merged mart_part_05.parquet
[07/32] merged mart_part_06.parquet
[08/32] merged mart_part_07.parquet
[09/32] merged mart_part_08.parquet
[10/32] merged mart_part_09.parquet
[11/32] merged mart_part_10.parquet
[12/32] merged mart_part_11.parquet
[13/32] merged mart_part_12.parquet
[14/32] merged mart_part_13.parquet
[15/32] merged mart_part_14.parquet
[16/32] merged mart_part_15.parquet
[17/32] merged mart_part_16.parquet
[18/32] merged mart_part_17.parquet
[19/32] merged mart_part_18.parquet
[20/32] merged mart_part_19.parquet
[21/32] merged mart_part_20.parquet
[22/32] merged mart_part_21.parquet
[23/32] merged mart_part_22.parquet
[24/32] merged mart_part_23.parquet
[25/32] merged mart_part_24.parquet
[26/32] merged mart_part_25.parquet
[27/32] merged mart_part_26.parquet
[28/32] merged mart_part_27.

## 9. Финальная короткая проверка

Не пересчитываем окна ещё раз по всему датасету.

Проверяем только итоговое число строк и основные распределения.

In [10]:
MART = sql_path(MART_PATH)

final_summary = con.sql(f"""
SELECT
    COUNT(*) AS events,
    COUNT(DISTINCT user_id) AS users,

    COUNT(*) FILTER (
        WHERE content_type_id = 0
    ) AS question_events,

    COUNT(*) FILTER (
        WHERE content_type_id = 1
    ) AS lecture_events,

    MIN(event_number) AS min_event_number,
    MIN(session_id) AS min_session_id

FROM read_parquet('{MART}')
""").df()

display(final_summary)

assert int(final_summary.loc[0, "events"]) == SOURCE_ROWS
assert int(final_summary.loc[0, "min_event_number"]) == 1
assert int(final_summary.loc[0, "min_session_id"]) == 1

print("✅ mart_events готова.")

,events,users,question_events,lecture_events,min_event_number,min_session_id
0,101230332,393656,99271300,1959032,1,1.0


✅ mart_events готова.


## 10. Пример

Выбираем одного пользователя и визуально убеждаемся, что события идут по времени и признаки используют только прошлое.

In [11]:
sample_user = con.sql(f"""
SELECT user_id
FROM read_parquet('{MART}')
LIMIT 1
""").fetchone()[0]

display(
    con.sql(f"""
    SELECT
        user_id,
        timestamp,
        content_kind,
        event_number,
        question_number,
        answered_correctly,
        previous_correct,
        rolling_accuracy_5,
        rolling_accuracy_20,
        questions_before,
        lectures_before,
        correct_answers_before,
        user_accuracy_before,
        error_streak,
        correct_streak,
        session_id
    FROM read_parquet('{MART}')
    WHERE user_id = {sample_user}
    ORDER BY event_number
    LIMIT 50
    """).df()
)

,user_id,timestamp,content_kind,event_number,question_number,answered_correctly,previous_correct,rolling_accuracy_5,rolling_accuracy_20,questions_before,lectures_before,correct_answers_before,user_accuracy_before,error_streak,correct_streak,session_id
0,40828,0,question,1,1.0,1,<NA>,NaN,NaN,0.0,0.0,0.0,NaN,0,0,1.0
1,40828,26382,question,2,2.0,0,1,1.000000,1.000000,1.0,0.0,1.0,1.000000,0,1,1.0
2,40828,51949,question,3,3.0,1,0,0.500000,0.500000,2.0,0.0,1.0,0.500000,1,0,1.0
3,40828,75274,question,4,4.0,1,1,0.666667,0.666667,3.0,0.0,2.0,0.666667,0,1,1.0
4,40828,174812,question,5,5.0,1,1,0.750000,0.750000,4.0,0.0,3.0,0.750000,0,2,1.0
5,40828,174812,question,6,6.0,0,1,0.800000,0.800000,5.0,0.0,4.0,0.800000,0,3,1.0
6,40828,174812,question,7,7.0,1,0,0.600000,0.666667,6.0,0.0,4.0,0.666667,1,0,1.0
7,40828,238072,question,8,8.0,0,1,0.800000,0.714286,7.0,0.0,5.0,0.714286,0,1,1.0
8,40828,238072,question,9,9.0,0,0,0.600000,0.625000,8.0,0.0,5.0,0.625000,1,0,1.0
9,40828,238072,question,10,10.0,0,0,0.400000,0.555556,9.0,0.0,5.0,0.555556,2,0,1.0
